# BCO7006 — Session 12
# APIs and HTTP Requests
### Getting data that doesn't come in a CSV · 2 hours

So far your data has arrived as a file. But the most useful data is often **live** (today's prices), **huge** (you want one slice), or **computed by someone else** (a service you call). For that, you talk to an **API** over HTTP.

**By the end of this session you will be able to:**
- Explain when an API beats a static file.
- Make a GET request with `requests` and read the status code.
- Add query parameters to ask the server for a subset.
- Navigate a JSON response (nested dicts and lists) to pull out fields.
- Load JSON results into a pandas DataFrame and analyse them.

> **Colab note:** `requests`, `json` and `pandas` are pre-installed. This notebook is built to run even with **no internet** — every live call falls back to a small cached file (`fakestore_products.json`) that ships alongside it. In Colab with internet, you'll hit the real API.

## 1 · Why an API, not a CSV?

A CSV is a snapshot. An API is a phone line to a live service. Use an API when:
- **The data changes fast** — stock prices, exchange rates. Re-downloading a file every minute is wasteful.
- **You want a slice of something enormous** — your own posts out of a billion, not the whole database.
- **Someone else does the computation** — a service classifies, geocodes, or converts for you.

Today's service is a public **online-store API** — the same Product / Price / Category shape you've worked with all term, but living on a server and arriving as JSON.

## 2 · The first request

The `requests` library is how Python talks HTTP. `requests.get(url)` fetches data and hands back a **response object**. The first thing to check is the **status code**: `200` means OK; anything starting with `4` or `5` is a problem.

In [1]:
import requests

API_URL = "https://fakestoreapi.com/products"

# Guarded so the notebook never crashes if there's no internet.
try:
    response = requests.get(API_URL, timeout=5)
    print("Status code:", response.status_code)   # 200 = OK
    print("Type of what we got back:", type(response))
except Exception as e:
    print("No live connection right now — we'll use cached data below.")
    print("(", type(e).__name__, ")")

Status code: 403
Type of what we got back: <class 'requests.models.Response'>


**Status codes you'll meet:**

| Code | Meaning |
|---|---|
| 200 | OK — here's your data |
| 301 | Moved — the endpoint changed |
| 400 | Bad request — you sent something wrong |
| 401 | Not authenticated — you need credentials |
| 403 | Forbidden — you're not allowed |
| 404 | Not found |
| 503 | Server not ready |

**Rule of thumb:** `2xx` good, `4xx` your fault, `5xx` their fault.

> **Trace.** If you requested `https://fakestoreapi.com/products/9999` (a product id that doesn't exist), what status code would you expect? Write it down before the peer-teaching activity.

## 3 · One helper, so the rest always works

To keep this notebook robust (and to teach the real-world habit of having a fallback), we wrap the fetch in a function: **try the live API, and if anything goes wrong, load the cached copy.** Everything after this uses `products`, which is guaranteed to be populated.

In [2]:
import json

def fetch_products():
    try:
        resp = requests.get(API_URL, timeout=5)
        resp.raise_for_status()        # turns a 4xx/5xx into an error we can catch
        return resp.json(), "live API"
    except Exception:
        with open("fakestore_products.json") as f:
            return json.load(f), "cached file"

products, source = fetch_products()
print(f"Loaded {len(products)} products from the {source}.")

Loaded 12 products from the cached file.


## 4 · JSON is just dicts and lists

`response.json()` (or `json.load`) hands you ordinary Python objects — here, a **list of dicts**. Navigating JSON is navigating dicts and lists, nothing new.

Look at one product:

In [3]:
first = products[0]          # a list -> index with [0]
print(type(first))           # dict
first

<class 'dict'>


{'id': 1,
 'title': 'Classic Cotton T-Shirt',
 'price': 22.3,
 'category': "men's clothing",
 'description': 'Soft everyday cotton tee.',
 'rating': {'rate': 3.9, 'count': 120}}

In [4]:
# Pull single fields out of the dict:
print("Title:   ", first['title'])
print("Price:   ", first['price'])
print("Category:", first['category'])
print("Rating:  ", first['rating']['rate'], "  <- a dict inside the dict")

Title:    Classic Cotton T-Shirt
Price:    22.3
Category: men's clothing
Rating:   3.9   <- a dict inside the dict


> ⏸️ **Pause and try.** Print the **title and price** of the **third** product (remember: lists start at index 0).

In [5]:
# Your turn: title and price of the third product.


## 5 · Query parameters — ask for a subset

You rarely want everything. Many APIs accept **query parameters** — extra instructions on the URL. `requests` builds them for you with `params=`. The store API supports `?limit=N`.

In [6]:
try:
    resp = requests.get(API_URL, params={"limit": 3}, timeout=5)
    print("URL requested:", resp.url)
    print("Got", len(resp.json()), "products back")
except Exception:
    print("Offline — but the idea: params={'limit': 3} appends '?limit=3' to the URL,")
    print("and the server returns only 3 products instead of all of them.")

URL requested: https://fakestoreapi.com/products?limit=3
Offline — but the idea: params={'limit': 3} appends '?limit=3' to the URL,
and the server returns only 3 products instead of all of them.


## 6 · JSON → DataFrame — back on familiar ground

A list of flat dicts drops straight into pandas. Now you can use everything from Sessions 9–11 on data you fetched yourself.

In [7]:
import pandas as pd

products_df = pd.DataFrame(products)
products_df[['id', 'title', 'price', 'category']].head()

,id,title,price,category
0,1,Classic Cotton T-Shirt,22.30,men's clothing
1,2,Slim Fit Jeans,55.99,men's clothing
2,3,Insulated Winter Jacket,109.95,men's clothing
3,4,Women's Knit Sweater,39.99,women's clothing
4,5,Summer Floral Dress,49.00,women's clothing


In [8]:
# The rating column is still a dict — flatten the bit we want:
products_df['rating_value'] = products_df['rating'].apply(lambda r: r['rate'])
products_df[['title', 'price', 'rating_value']].head()

,title,price,rating_value
0,Classic Cotton T-Shirt,22.30,3.9
1,Slim Fit Jeans,55.99,4.1
2,Insulated Winter Jacket,109.95,4.7
3,Women's Knit Sweater,39.99,3.5
4,Summer Floral Dress,49.00,4.2


## 7 · Mini-project — analyse the fetched data (≈15 min)

Use `products_df`. For each task, write the code and a one-line finding.

**Q1.** What is the **average price per category**? (Reuse your Session 11 skills — a `groupby` and a bar chart.)
**Q2.** Which single product has the **highest rating value**?
**Q3.** How many products cost **more than $100**?
**Open (Extend).** Pick a *different* no-auth public API (your peer-teaching one is fine), fetch it, turn it into a DataFrame, and show one fact about it.

In [9]:
# Q1 — average price per category (+ a bar chart)


In [10]:
# Q2 — highest-rated product


In [11]:
# Q3 — count of products over $100


In [12]:
# Open — fetch a different API into a DataFrame and show one fact


## 8 · AI critique (≈10 min)

**An LLM was asked: "get the products from the API and print the first title." It returned this:**

```python
import requests
data = requests.get("https://fakestoreapi.com/products")
print(data[0]["title"])
```

This looks reasonable and **will crash**. In the cell below, write what's wrong (at least two issues) and a corrected version.

*(Hints: what does `requests.get` actually return — the data, or an object wrapping it? What step is missing before you can index it like a list? What if the request fails?)*

In [13]:
# Your critique (as comments) + corrected version.
# What's wrong:
# 1.
# 2.

# Corrected:


### Reflection — required
If you used an AI tool here, paste your prompt(s) and one line on what you changed. If not, say so.

In [14]:
# Prompt used:
# What I changed:


## ✅ Before you leave
- [ ] Notebook runs top-to-bottom with no errors (Runtime ▸ Restart and run all)
- [ ] You can explain what a `200` status code means and what `response.json()` returns
- [ ] All three mini-project questions have code **and** a one-line finding
- [ ] AI reflection cell is filled in

**That's the unit.** You can now write classes, work with NumPy and pandas, visualise data, and pull live data from the web — the full toolkit of a coding business analyst.